# Customer 360 Intelligence & Churn Prediction Platform
## Notebook 01: Olist Dataset Ingestion, Quality Audit & Exploratory Data Analysis (EDA)

**Client / Context:** Deloitte Analytics Consulting Practice  
**Objective:** Systematic discovery, structural integrity auditing, data dictionary generation, and exploratory visualizations across 9 relational Olist e-commerce datasets.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set presentation styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 130

DATA_DIR = os.path.join('..', 'data', 'raw')
FIGURES_DIR = os.path.join('..', 'reports', 'figures')

### 1. Ingestion and Structural Inspection
Loading all 9 CSV files and capturing dimensions, missing percentages, and primary-key viability.

In [2]:
DATASET_FILES = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {name: pd.read_csv(os.path.join(DATA_DIR, fname)) for name, fname in DATASET_FILES.items()}
summary_data = []
for name, df in dfs.items():
    summary_data.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Duplicates': df.duplicated().sum(),
        'Missing Cells': df.isnull().sum().sum(),
        'Missing %': round((df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100, 2)
    })
pd.DataFrame(summary_data)

### 2. Relational Cross-Table Integrity & Entity Resolution
- Customer granularity (`customer_id` vs `customer_unique_id`)
- Referential consistency across Orders, Items, Payments, and Reviews

In [3]:
total_sessions = dfs['customers']['customer_id'].nunique()
total_unique_customers = dfs['customers']['customer_unique_id'].nunique()
repeat_buyer_count = total_sessions - total_unique_customers
repeat_rate = (repeat_buyer_count / total_unique_customers) * 100

print(f"Total Order Sessions: {total_sessions:,}")
print(f"Total Unique Natural Persons: {total_unique_customers:,}")
print(f"Repeat Buyer Events: {repeat_buyer_count:,} ({repeat_rate:.2f}% raw repeat rate)")

### 3. Key Findings & Exploratory Visualizations
Generated visuals and analytical summaries saved to `reports/figures/`.

In [4]:
# Visualizing Order Status Distribution
plt.figure(figsize=(10, 4))
sns.countplot(data=dfs['orders'], x='order_status', order=dfs['orders']['order_status'].value_counts().index, palette='Blues_r')
plt.title('Order Volume by Order Status', fontsize=12, fontweight='bold')
plt.xlabel('Order Status')
plt.ylabel('Order Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()